# Segmentación de naranjas con YOLO26n-seg

Notebook para probar **YOLO26n-seg** (preentrenado en COCO) en la segmentación de naranjas,
medir qué tan bien anda con el dataset *Orange occlusion* de Roboflow y exportarlo a **ONNX**
para usarlo en la web de ThreeDetection.

**Flujo:** GPU → entorno → dataset → prueba visual → mAP → ONNX para la web → *(opcional)* transfer learning y fine-tuning.

## 0. Antes de empezar

1. **GPU:** menú *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU*.
2. **API key de Roboflow:** en la barra izquierda, ícono de la llave (🔑 *Secrets*), agregar
   `ROBOFLOW_API_KEY` con tu clave y activar *Acceso del notebook*.
3. *Entorno de ejecución → Ejecutar todas*.

Los resultados (métricas, gráficos, modelos y el ONNX para la web) quedan en
**Google Drive**, en `MyDrive/naranjas/`. El dataset se descarga en `/content` porque ahí se lee más rápido.

## 1. GPU

In [ ]:
!nvidia-smi

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        'No hay GPU. Andá a "Entorno de ejecución > Cambiar tipo de entorno de ejecución", '
        'elegí "T4 GPU" y volvé a ejecutar el notebook.'
    )
print('GPU:', torch.cuda.get_device_name(0))

## 2. Entorno

Se monta Drive, se instalan las librerías y se definen las rutas y parámetros generales.

In [ ]:
!pip install -q ultralytics roboflow onnx onnxruntime onnxslim

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')

DRIVE_DIR = Path('/content/drive/MyDrive/naranjas')   # resultados que se conservan
WORK_DIR = Path('/content')                            # dataset y archivos temporales

In [ ]:
import json
import shutil

import cv2
import matplotlib.pyplot as plt
import numpy as np
import onnxruntime as ort
import pandas as pd
import ultralytics
import yaml
from IPython.display import display
from ultralytics import YOLO

ultralytics.checks()

DRIVE_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR = DRIVE_DIR / 'runs'      # gráficos de validación y entrenamientos

IMGSZ = 640          # tamaño de entrada del modelo (también el de la web)
CONF = 0.25          # confianza mínima para contar una naranja
COCO_ORANGE = 49     # índice de la clase "orange" en COCO
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

## 3. Dataset

Se descarga *Orange occlusion dataset* (versión 5) en formato YOLO. Roboflow arma un `data.yaml`
con rutas relativas que a veces fallan, así que se reescribe con rutas absolutas.

In [ ]:
from google.colab import userdata
from roboflow import Roboflow

rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
dataset = (
    rf.workspace('canerbeldek')
    .project('orange-occlusion-dataset')
    .version(5)
    .download('yolov8', location=str(WORK_DIR / 'orange-occlusion'))
)
DATASET_DIR = Path(dataset.location)

In [ ]:
original = yaml.safe_load((DATASET_DIR / 'data.yaml').read_text())
nombres = original['names']
CLASES_DATASET = dict(enumerate(nombres)) if isinstance(nombres, list) else {int(k): v for k, v in nombres.items()}
print('Clases del dataset:', CLASES_DATASET)

# Si el dataset distingue tipos de naranja (por ejemplo, ocluida / visible), para nosotros
# todas son "naranja": se entrena con una sola clase y en la evaluación de COCO todas pasan a la 49.
UNA_CLASE = len(CLASES_DATASET) > 1

# Roboflow llama "valid" a la carpeta de validación.
SPLITS = {
    split: carpeta
    for split, carpeta in [('train', 'train'), ('val', 'valid'), ('test', 'test')]
    if (DATASET_DIR / carpeta / 'images').is_dir()
}
if 'test' not in SPLITS:
    print('⚠️ El dataset no trae split de test: se evalúa sobre validación.')
    SPLITS['test'] = SPLITS['val']

DATA_YAML = DATASET_DIR / 'data_local.yaml'
DATA_YAML.write_text(yaml.safe_dump({
    'path': str(DATASET_DIR),
    'train': f"{SPLITS['train']}/images",
    'val': f"{SPLITS['val']}/images",
    'test': f"{SPLITS['test']}/images",
    'names': CLASES_DATASET,
}, allow_unicode=True))

TEST_IMAGES = DATASET_DIR / SPLITS['test'] / 'images'
TEST_LABELS = DATASET_DIR / SPLITS['test'] / 'labels'
print(DATA_YAML.read_text())

### Conteo por split y tipo de etiqueta

Cada línea de una etiqueta YOLO es un objeto. Si tiene 5 valores es una **caja**
(`clase x y ancho alto`); si tiene más, es un **polígono** (`clase x1 y1 x2 y2 ...`).
Para medir el mAP de máscara y para entrenar segmentación hacen falta polígonos.

In [ ]:
def imagenes(carpeta):
    return sorted(p for p in Path(carpeta).iterdir() if p.suffix.lower() in IMG_EXT)


def leer_etiquetas(ruta_txt):
    """Devuelve una lista de (clase, coordenadas normalizadas) por objeto."""
    if not ruta_txt.exists():
        return []
    objetos = []
    for linea in ruta_txt.read_text().splitlines():
        valores = linea.split()
        if len(valores) >= 5:
            objetos.append((int(valores[0]), np.array(valores[1:], dtype=float)))
    return objetos


def etiqueta_de(imagen):
    return imagen.parent.parent / 'labels' / f'{imagen.stem}.txt'


filas = []
tipos = {'cajas': 0, 'poligonos': 0}
for split, carpeta in SPLITS.items():
    imgs = imagenes(DATASET_DIR / carpeta / 'images')
    objetos = [obj for img in imgs for obj in leer_etiquetas(etiqueta_de(img))]
    for _, coords in objetos:
        tipos['cajas' if len(coords) == 4 else 'poligonos'] += 1
    filas.append({'split': split, 'carpeta': carpeta, 'imagenes': len(imgs), 'naranjas': len(objetos),
                  'naranjas por imagen': round(len(objetos) / max(len(imgs), 1), 1)})

display(pd.DataFrame(filas))
print('Tipos de etiqueta:', tipos)

HAY_POLIGONOS = tipos['poligonos'] > 0
if not HAY_POLIGONOS:
    print('⚠️ El dataset solo tiene cajas: se evalúan cajas y el mAP de máscara queda vacío.')
elif tipos['cajas']:
    print('⚠️ El dataset mezcla cajas y polígonos: Ultralytics va a descartar los polígonos.')

In [ ]:
def dibujar_reales(imagen):
    """Imagen RGB con las naranjas etiquetadas pintadas encima."""
    img = cv2.cvtColor(cv2.imread(str(imagen)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    capa = img.copy()
    for _, coords in leer_etiquetas(etiqueta_de(imagen)):
        if len(coords) == 4:
            cx, cy, bw, bh = coords * [w, h, w, h]
            p1 = (int(cx - bw / 2), int(cy - bh / 2))
            p2 = (int(cx + bw / 2), int(cy + bh / 2))
            cv2.rectangle(capa, p1, p2, (255, 140, 0), -1)
            cv2.rectangle(img, p1, p2, (255, 220, 120), 2)
        else:
            puntos = (coords.reshape(-1, 2) * [w, h]).astype(np.int32)
            cv2.fillPoly(capa, [puntos], (255, 140, 0))
            cv2.polylines(img, [puntos], True, (255, 220, 120), 2)
    return cv2.addWeighted(capa, 0.45, img, 0.55, 0)


def grilla(imagenes_rgb, titulos, columnas=4):
    filas_g = (len(imagenes_rgb) + columnas - 1) // columnas
    fig, ejes = plt.subplots(filas_g, columnas, figsize=(4 * columnas, 4 * filas_g), squeeze=False)
    for eje in ejes.flat:
        eje.axis('off')
    for eje, img, titulo in zip(ejes.flat, imagenes_rgb, titulos):
        eje.imshow(img)
        eje.set_title(titulo, fontsize=10)
    plt.tight_layout()
    plt.show()


imagenes_test = imagenes(TEST_IMAGES)
muestra = imagenes_test[:8]
grilla([dibujar_reales(p) for p in muestra],
       [f'{p.name[:22]}\n{len(leer_etiquetas(etiqueta_de(p)))} naranjas reales' for p in muestra])

## 4. Modelo base: prueba visual

YOLO26n-seg preentrenado en COCO ya conoce la clase **orange** (índice 49). Sin entrenar nada,
se le pide que segmente solo esa clase y se compara cuántas naranjas encuentra contra las etiquetadas.

In [ ]:
modelo_base = YOLO('yolo26n-seg.pt')
print('Clase', COCO_ORANGE, '=', modelo_base.names[COCO_ORANGE])


def conteo_por_imagen(modelo, clases=None, imgsz=IMGSZ, ejemplos=0):
    """Naranjas predichas vs. reales en cada imagen de test, y algunas imágenes dibujadas."""
    filas_c, dibujos = [], []
    for r in modelo.predict(imagenes_test, classes=clases, conf=CONF, imgsz=imgsz, stream=True, verbose=False):
        ruta = Path(r.path)
        filas_c.append({'imagen': ruta.name,
                        'reales': len(leer_etiquetas(etiqueta_de(ruta))),
                        'predichas': len(r.boxes)})
        if len(dibujos) < ejemplos:
            dibujos.append((r.plot()[..., ::-1], filas_c[-1]))
    tabla = pd.DataFrame(filas_c)
    tabla['error'] = tabla['predichas'] - tabla['reales']
    return tabla, dibujos


conteo_base, dibujos = conteo_por_imagen(modelo_base, clases=[COCO_ORANGE], ejemplos=8)
grilla([img for img, _ in dibujos],
       [f"{f['imagen'][:22]}\npredichas {f['predichas']} / reales {f['reales']}" for _, f in dibujos])

MAE_BASE = conteo_base['error'].abs().mean()
print(f'Error absoluto medio del conteo: {MAE_BASE:.2f} naranjas por imagen')
print(f"Predichas en total: {conteo_base['predichas'].sum()} | reales: {conteo_base['reales'].sum()}")
conteo_base.to_csv(DRIVE_DIR / 'conteo_base.csv', index=False)
display(conteo_base.head(15))

## 5. Modelo base: mAP

El modelo predice con los índices de COCO (naranja = 49) y el dataset etiqueta las naranjas como 0.
Para que Ultralytics compare naranja con naranja se copian las imágenes y etiquetas de test
**cambiando la clase a 49**, con un `data_coco.yaml` que tiene los 80 nombres de COCO.

> `val()` no filtra por `classes` (ese argumento es de `predict()`), pero no hace falta:
> el mAP se promedia solo sobre las clases que aparecen en las etiquetas reales, es decir, solo
> la 49. Si el modelo confunde una naranja con otra clase (por ejemplo `apple`), cuenta como
> naranja no encontrada.

In [ ]:
COCO_DIR = WORK_DIR / 'coco_eval'
shutil.rmtree(COCO_DIR, ignore_errors=True)   # borra también el labels.cache viejo
shutil.copytree(TEST_IMAGES, COCO_DIR / 'images')
(COCO_DIR / 'labels').mkdir(parents=True)

for imagen in imagenes(COCO_DIR / 'images'):
    lineas = [f"{COCO_ORANGE} {' '.join(f'{v:.6g}' for v in coords)}"
              for _, coords in leer_etiquetas(TEST_LABELS / f'{imagen.stem}.txt')]
    (COCO_DIR / 'labels' / f'{imagen.stem}.txt').write_text('\n'.join(lineas))

COCO_YAML = COCO_DIR / 'data_coco.yaml'
COCO_YAML.write_text(yaml.safe_dump({
    'path': str(COCO_DIR), 'train': 'images', 'val': 'images', 'test': 'images',
    'names': modelo_base.names,
}, allow_unicode=True))
print(f'{len(imagenes(COCO_DIR / "images"))} imágenes de test remapeadas a la clase {COCO_ORANGE}.')

In [ ]:
def resumen(nombre, metricas, mae=np.nan):
    """Métricas principales de una corrida de val() en una fila."""
    fila = {
        'modelo': nombre,
        'caja mAP50': metricas.box.map50,
        'caja mAP50-95': metricas.box.map,
        'máscara mAP50': metricas.seg.map50 if HAY_POLIGONOS else np.nan,
        'máscara mAP50-95': metricas.seg.map if HAY_POLIGONOS else np.nan,
        'precisión': metricas.box.mp,
        'recall': metricas.box.mr,
        'MAE conteo': mae,
    }
    return {k: round(float(v), 4) if isinstance(v, (float, np.floating)) else v for k, v in fila.items()}


metricas_base = modelo_base.val(data=str(COCO_YAML), split='test', imgsz=IMGSZ, batch=16,
                                project=str(RUNS_DIR), name='val_base', exist_ok=True)

tabla_base = pd.DataFrame([resumen('Base (COCO, clase 49)', metricas_base, MAE_BASE)])
tabla_base.to_csv(DRIVE_DIR / 'metricas_base.csv', index=False)
display(tabla_base)

**Cómo leerlo:** *mAP50* mide si encuentra las naranjas con una superposición de al menos 50 %;
*mAP50-95* exige además bordes precisos. *Precisión* = de lo que marcó, cuánto era naranja;
*recall* = de las naranjas reales, cuántas encontró. Si el recall es bajo (típico con naranjas
chicas o tapadas por hojas), conviene pasar a la sección 7.

## 6. Exportar a ONNX para la web

Se exporta con `nms=True`: el ONNX trae el NMS adentro y devuelve hasta 300 detecciones ya filtradas,
así la web no tiene que hacer NMS en JavaScript.

> **¿Por qué no `nms=False`?** YOLO26 tiene dos cabezas: una *uno-a-muchos* que necesita NMS y otra
> *uno-a-uno* que no lo necesita. `predict()` y `val()` del `.pt` usan la primera, y `nms=False` exporta
> la segunda, que da otras confianzas. Con `nms=True` el ONNX usa la misma cabeza que se midió en la
> sección 5, así que el mAP de arriba es el del modelo que corre en la web. Las dos salidas tienen el
> mismo formato y la web acepta cualquiera de los dos.

In [ ]:
onnx_base = Path(modelo_base.export(format='onnx', imgsz=IMGSZ, nms=True, conf=CONF, device='cpu'))

sesion = ort.InferenceSession(str(onnx_base), providers=['CPUExecutionProvider'])
for e in sesion.get_inputs():
    print('Entrada:', e.name, e.shape)
for s in sesion.get_outputs():
    print('Salida: ', s.name, s.shape)
print('\n[1, 300, 38] = 300 detecciones x (x1, y1, x2, y2, confianza, clase, 32 coeficientes de máscara)')
print('[1, 32, 160, 160] = 32 prototipos de máscara; cada máscara es sigmoide(coeficientes · prototipos)')

In [ ]:
# Predicción de control: el .onnx tiene que contar lo mismo que el .pt.
# rect=False hace que el .pt también rellene la imagen hasta 640x640, como el ONNX y la web.
# El ONNX corre en CPU como en la web (y así Ultralytics no intenta instalar onnxruntime-gpu).
modelo_onnx = YOLO(str(onnx_base), task='segment')
for imagen in imagenes_test[:3]:
    n_pt = len(modelo_base.predict(imagen, classes=[COCO_ORANGE], conf=CONF, imgsz=IMGSZ, rect=False,
                                   verbose=False)[0].boxes)
    n_onnx = len(modelo_onnx.predict(imagen, classes=[COCO_ORANGE], conf=CONF, imgsz=IMGSZ, device='cpu',
                                     verbose=False)[0].boxes)
    print(f"{imagen.name[:30]:30}  .pt: {n_pt:3}  .onnx: {n_onnx:3}  {'✅' if n_pt == n_onnx else '⚠️'}")

In [ ]:
def guardar_para_web(ruta_onnx, carpeta, nombre, class_id, imgsz, names):
    """Copia el ONNX y escribe el model.json que lee la web (js/demo/segmenter.js)."""
    carpeta.mkdir(parents=True, exist_ok=True)
    shutil.copy(ruta_onnx, carpeta / 'orange-seg.onnx')
    s = ort.InferenceSession(str(ruta_onnx), providers=['CPUExecutionProvider'])
    info = {
        'name': nombre,
        'file': 'orange-seg.onnx',
        'imgsz': imgsz,
        'classId': class_id,
        'conf': CONF,
        'input': s.get_inputs()[0].name,
        'outputs': [{'name': o.name, 'shape': o.shape} for o in s.get_outputs()],
        'names': {str(k): v for k, v in names.items()},
    }
    (carpeta / 'model.json').write_text(json.dumps(info, indent=2, ensure_ascii=False))
    tamano = (carpeta / 'orange-seg.onnx').stat().st_size / 1e6
    print(f'Guardado en {carpeta}: orange-seg.onnx ({tamano:.1f} MB) y model.json')


guardar_para_web(onnx_base, DRIVE_DIR / 'web', 'YOLO26n-seg (COCO, clase orange)',
                 COCO_ORANGE, IMGSZ, modelo_base.names)

Para usarlo en la web, copiá `MyDrive/naranjas/web/orange-seg.onnx` y `model.json` a
`ThreeDetection/model/` (ver el README del repo).

## 7. Transfer learning y fine-tuning (preparado)

Esta sección queda escrita pero **no se ejecuta** mientras `ENTRENAR = False`. Sirve si el modelo base
no alcanza. Se entrena sobre el dataset original, donde la naranja es la **clase 0**:

1. **Transfer learning:** se congelan las primeras 10 capas (el *backbone*) y solo se entrena la cabeza.
2. **Fine-tuning:** se parte del mejor modelo de transfer, se descongela todo y se reentrena con
   `AdamW`, `lr0=1e-4` e imágenes de 1024 px (las naranjas chicas o tapadas se ven mejor), con
   augmentations de color y brillo para tolerar distintas luces.

Los entrenamientos se guardan en Drive: si Colab se desconecta, el último checkpoint queda en
`runs/<nombre>/weights/last.pt` y se puede retomar con `YOLO(ruta).train(resume=True)`.

In [ ]:
ENTRENAR = False

EPOCAS_TRANSFER = 30
EPOCAS_FT = 30
IMGSZ_FT = 1024      # el ONNX del fine-tuning se exporta con este tamaño (la web lo lee de model.json)
BATCH = 16
BATCH_FT = 8         # a 1024 px entra menos en la memoria de la T4
PACIENCIA = 10       # épocas sin mejora antes de cortar

In [ ]:
if ENTRENAR:
    modelo_transfer = YOLO('yolo26n-seg.pt')
    modelo_transfer.train(
        data=str(DATA_YAML), epochs=EPOCAS_TRANSFER, imgsz=IMGSZ, batch=BATCH,
        freeze=10, patience=PACIENCIA, single_cls=UNA_CLASE,
        project=str(RUNS_DIR), name='transfer', exist_ok=True,
    )
    MEJOR_TRANSFER = Path(modelo_transfer.trainer.best)
    print('Mejor modelo de transfer:', MEJOR_TRANSFER)
else:
    print('ENTRENAR = False: se saltea el transfer learning.')

In [ ]:
if ENTRENAR:
    modelo_ft = YOLO(MEJOR_TRANSFER)
    modelo_ft.train(
        data=str(DATA_YAML), epochs=EPOCAS_FT, imgsz=IMGSZ_FT, batch=BATCH_FT,
        optimizer='AdamW', lr0=1e-4, warmup_epochs=0, patience=PACIENCIA, single_cls=UNA_CLASE,
        hsv_h=0.01,   # tono: poco, para que una naranja no pase a parecer un limón
        hsv_s=0.7,    # saturación
        hsv_v=0.5,    # brillo: sol directo, sombra, nublado
        project=str(RUNS_DIR), name='finetune', exist_ok=True,
    )
    MEJOR_FT = Path(modelo_ft.trainer.best)
    print('Mejor modelo de fine-tuning:', MEJOR_FT)
else:
    print('ENTRENAR = False: se saltea el fine-tuning.')

### Comparación contra el modelo base

Los modelos entrenados se evalúan sobre el mismo split de test, ahora con el dataset original (clase 0).

In [ ]:
if ENTRENAR:
    filas_comp = [tabla_base.iloc[0].to_dict()]
    for nombre, pesos, imgsz in [('Transfer learning', MEJOR_TRANSFER, IMGSZ), ('Fine-tuning', MEJOR_FT, IMGSZ_FT)]:
        modelo = YOLO(pesos)
        metricas = modelo.val(data=str(DATA_YAML), split='test', imgsz=imgsz, batch=BATCH_FT,
                              single_cls=UNA_CLASE, project=str(RUNS_DIR), name=f'val_{pesos.parent.parent.name}',
                              exist_ok=True)
        conteo, _ = conteo_por_imagen(modelo, imgsz=imgsz)
        filas_comp.append(resumen(nombre, metricas, conteo['error'].abs().mean()))

    comparacion = pd.DataFrame(filas_comp)
    comparacion.to_csv(DRIVE_DIR / 'metricas_comparacion.csv', index=False)
    display(comparacion)
else:
    print('ENTRENAR = False: no hay modelos entrenados para comparar.')

In [ ]:
if ENTRENAR:
    modelo_ft = YOLO(MEJOR_FT)
    _, dib_base = conteo_por_imagen(modelo_base, clases=[COCO_ORANGE], ejemplos=4)
    _, dib_ft = conteo_por_imagen(modelo_ft, imgsz=IMGSZ_FT, ejemplos=4)
    grilla([img for img, _ in dib_base] + [img for img, _ in dib_ft],
           [f"Base: {f['predichas']} / {f['reales']}" for _, f in dib_base]
           + [f"Fine-tuning: {f['predichas']} / {f['reales']}" for _, f in dib_ft])

In [ ]:
if ENTRENAR:
    onnx_ft = Path(YOLO(MEJOR_FT).export(format='onnx', imgsz=IMGSZ_FT, nms=True, conf=CONF, device='cpu'))
    guardar_para_web(onnx_ft, DRIVE_DIR / 'web_finetune', 'YOLO26n-seg ajustado a naranjas',
                     0, IMGSZ_FT, {0: 'orange'})
    print('Para usarlo en la web, copiá MyDrive/naranjas/web_finetune/ a ThreeDetection/model/.')

## Próximos pasos

- Copiar `MyDrive/naranjas/web/` (o `web_finetune/`) a `ThreeDetection/model/` y probar la demo.
- Comparar el conteo de la web con `conteo_base.csv` usando las mismas imágenes de test.
- Si el modelo base no alcanza, poner `ENTRENAR = True` y volver a ejecutar la sección 7.